# Week 45, Day 3: Depth-first traversals

Companion notebook. The interactive figures, X-ray views, quizzes and auto-marked exercises are on the [lesson page](https://peter-adepoju.github.io/python-atlas/days/w045d3.html). Use this notebook to keep your own copy of the code.

**How to use it:** run each cell with Shift+Enter. Change the code and run it again: that's how you learn.

In [ ]:
# Run this cell once. It lets you check exercise answers inside this notebook.
# Workflow: run your exercise cell, then immediately run the "check" cell below it.
import io, builtins

def check(tests_src):
    """Run an exercise's tests against the cell you ran just before this one."""
    ip = get_ipython()
    code = ip.user_ns["In"][-2]
    def run(feed=(), echo=False):
        out = io.StringIO(); q = [str(x) for x in feed]; real_in, real_print = builtins.input, builtins.print
        def fake_input(p=""):
            if not q:
                raise AssertionError("The program asked for more input than the test gave it.")
            v = q.pop(0); out.write(str(p) + (v + "\n" if echo else "")); return v
        def fake_print(*a, **k):
            k.setdefault("file", out); real_print(*a, **k)
        builtins.input, builtins.print = fake_input, fake_print
        try:
            exec(code, {"__name__": "__main__"})
        finally:
            builtins.input, builtins.print = real_in, real_print
        return out.getvalue()
    norm = lambda t: "\n".join(l.rstrip() for l in str(t).rstrip().splitlines())
    def expect_output(want, inputs=(), echo=False):
        got = run(inputs, echo)
        assert norm(got) == norm(want), f"expected {norm(want)!r} but got {norm(got)!r}"
    import re
    uses = lambda *names: all(re.search(r"\b" + re.escape(n) + r"\b", code) for n in names)
    def run_with(subs, feed=(), echo=False):
        nonlocal code
        saved = code
        for old, new in subs.items():
            code = code.replace(old, new, 1)
        try:
            return run(feed, echo)
        finally:
            code = saved
    env = dict(ip.user_ns, run=run, norm=norm, source=code, expect_output=expect_output, uses=uses, run_with=run_with)
    try:
        env["output"] = run()
    except AssertionError:
        env["output"] = ""
    exec(tests_src, env)
    tests = [(k, v) for k, v in env.items() if k.startswith("test_") and callable(v)]
    ok = 0
    for name, fn in tests:
        label = (fn.__doc__ or name[5:].replace("_", " ")).strip()
        try:
            fn(); ok += 1; print("✓", label)
        except AssertionError as e:
            print("✗", label + ":", e or "not quite yet")
    print(f"{ok} of {len(tests)} checks passed.")


**Predict before you run:** what will this print?



In [ ]:
class TreeNode:
    def __init__(self, val, left=None, right=None):
        self.val, self.left, self.right = val, left, right

def walk(node, out, order):
    if node is None:
        return
    if order == "pre":
        out.append(node.val)
    walk(node.left, out, order)
    if order == "in":
        out.append(node.val)
    walk(node.right, out, order)
    if order == "post":
        out.append(node.val)

t = TreeNode("*", TreeNode("+", TreeNode(2), TreeNode(3)), TreeNode(4))
for order in ["pre", "in", "post"]:
    out = []
    walk(t, out, order)
    print(order, " ".join(map(str, out)))

In [ ]:
class TreeNode:
    def __init__(self, val, left=None, right=None):
        self.val, self.left, self.right = val, left, right

def preorder(root):
    out, stack = [], [root] if root else []
    while stack:
        node = stack.pop()
        out.append(node.val)
        if node.right:
            stack.append(node.right)
        if node.left:
            stack.append(node.left)
    return out

def inorder(root):
    out, stack, node = [], [], root
    while stack or node:
        while node:                      # go as far left as possible
            stack.append(node)
            node = node.left
        node = stack.pop()
        out.append(node.val)
        node = node.right                # then do the right subtree the same way
    return out

def postorder(root):
    out, stack = [], [root] if root else []
    while stack:
        node = stack.pop()
        out.append(node.val)             # node, right, left ...
        if node.left:
            stack.append(node.left)
        if node.right:
            stack.append(node.right)
    return out[::-1]                     # ... reversed: left, right, node

T = TreeNode
t = T("F", T("B", T("A"), T("D", T("C"), T("E"))), T("G", None, T("I", T("H"))))
print("".join(preorder(t)), "".join(inorder(t)), "".join(postorder(t)))

chain = None
for i in range(5000):                    # far too tall for recursion
    chain = T(i, chain)
print(len(inorder(chain)), inorder(chain)[:3])

In [ ]:
class TreeNode:
    def __init__(self, val, left=None, right=None):
        self.val, self.left, self.right = val, left, right

class InOrder:
    def __init__(self, root):
        self._stack = []
        self._push_left(root)

    def _push_left(self, node):
        while node is not None:
            self._stack.append(node)
            node = node.left

    def __iter__(self):
        return self

    def __next__(self):
        if not self._stack:
            raise StopIteration
        node = self._stack.pop()
        self._push_left(node.right)
        return node.val

T = TreeNode
t = T(4, T(2, T(1), T(3)), T(6, T(5), T(7)))
it = InOrder(t)
print(next(it), next(it), list(it))
print(list(zip(InOrder(t), InOrder(T(10, T(5))))))

In [ ]:
class TreeNode:
    def __init__(self, val, left=None, right=None):
        self.val, self.left, self.right = val, left, right

def build(preorder, inorder):
    if not preorder:
        return None
    root_val = preorder[0]
    k = inorder.index(root_val)                  # k values are in the left subtree
    return TreeNode(root_val,
                    build(preorder[1:k + 1], inorder[:k]),
                    build(preorder[k + 1:], inorder[k + 1:]))

def show(node):
    return "." if node is None else f"({show(node.left)} {node.val} {show(node.right)})"

print(show(build(list("FBADCEGIH"), list("ABCDEFGHI"))))

In [ ]:
class TreeNode:
    def __init__(self, val, left=None, right=None):
        self.val, self.left, self.right = val, left, right

def save(node, out):
    if node is None:
        out.append("#")
        return
    out.append(str(node.val))
    save(node.left, out)
    save(node.right, out)

def load(tokens, position):
    """Build the subtree starting at tokens[position[0]], moving position on as tokens are used."""
    token = tokens[position[0]]
    position[0] += 1
    if token == "#":
        return None
    node = TreeNode(int(token))
    node.left = load(tokens, position)
    node.right = load(tokens, position)
    return node

out = []
save(TreeNode(1, TreeNode(2), TreeNode(3, TreeNode(4), TreeNode(5))), out)
text = " ".join(out)
print(text)
again = load(text.split(), [0])
out2 = []
save(again, out2)
print(" ".join(out2) == text)

> **Key idea.** Depth-first traversals of a binary tree: pre-order visits node, left, right (copying, saving, prefix); in-order visits left, node, right (sorted order in search trees, infix); post-order visits left, right, node (sizes, deletion, evaluation, postfix). Each is O(n) time, O(h) space. With an explicit stack: pre-order pops a node and pushes right then left; in-order pushes the left spine, pops, then moves right; post-order is a node-right-left walk, reversed. An iterator class can produce the in-order values one at a time. Pre-order plus in-order (distinct values) rebuild a tree; pre-order with `#` markers saves and loads one.

### Exercise: Three orders, recursively

Write `preorder(root)`, `inorder(root)` and `postorder(root)`, each returning a list of values, recursively, using a helper that appends to one list (rather than joining many lists together).

In [ ]:
class TreeNode:
    def __init__(self, val, left=None, right=None):
        self.val, self.left, self.right = val, left, right

def preorder(root):
    ___

def inorder(root):
    ___

def postorder(root):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("T = TreeNode\ndef test_orders():\n    \"three orders\"\n    t = T(\"F\", T(\"B\", T(\"A\"), T(\"D\", T(\"C\"), T(\"E\"))), T(\"G\", None, T(\"I\", T(\"H\"))))\n    assert \"\".join(preorder(t)) == \"FBADCEGIH\" and \"\".join(inorder(t)) == \"ABCDEFGHI\" and \"\".join(postorder(t)) == \"ACEDBHIGF\"\n    assert preorder(None) == inorder(None) == postorder(None) == []")

### Exercise: Pre-order and in-order with a stack

Write `preorder(root)` and `inorder(root)` **without recursion**, using an explicit stack. The checks walk a chain 20,000 nodes tall.

In [ ]:
class TreeNode:
    def __init__(self, val, left=None, right=None):
        self.val, self.left, self.right = val, left, right

def preorder(root):
    ___

def inorder(root):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("import ast\nT = TreeNode\ndef test_orders():\n    \"orders\"\n    t = T(\"F\", T(\"B\", T(\"A\"), T(\"D\", T(\"C\"), T(\"E\"))), T(\"G\", None, T(\"I\", T(\"H\"))))\n    assert \"\".join(preorder(t)) == \"FBADCEGIH\" and \"\".join(inorder(t)) == \"ABCDEFGHI\" and preorder(None) == [] and inorder(None) == []\ndef test_tall():\n    \"a tall tree\"\n    left_chain = right_chain = None\n    for i in range(20_000):\n        left_chain = T(i, left_chain)\n        right_chain = T(i, None, right_chain)\n    assert inorder(left_chain)[:3] == [0, 1, 2] and preorder(right_chain)[:3] == [19_999, 19_998, 19_997]\n    fns = [n for n in ast.parse(source).body if isinstance(n, ast.FunctionDef)]\n    for f in fns:\n        assert not any(isinstance(c, ast.Call) and getattr(c.func, \"id\", \"\") == f.name for c in ast.walk(f)), f\"{f.name} must not call itself\"")

### Exercise: Post-order, two ways

Write `postorder_reversed(root)` with the trick from today (a node-right-left walk with one stack, then reverse), and `postorder_marked(root)`, which instead pushes `(node, expanded)` pairs: a node is pushed first with `expanded=False`; when popped unexpanded, push it back as expanded, then its right child, then its left child (each unexpanded); when popped expanded, visit it. Neither may use recursion.

In [ ]:
class TreeNode:
    def __init__(self, val, left=None, right=None):
        self.val, self.left, self.right = val, left, right

def postorder_reversed(root):
    ___

def postorder_marked(root):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("T = TreeNode\ndef test_post():\n    \"post-order\"\n    t = T(\"F\", T(\"B\", T(\"A\"), T(\"D\", T(\"C\"), T(\"E\"))), T(\"G\", None, T(\"I\", T(\"H\"))))\n    assert \"\".join(postorder_reversed(t)) == \"ACEDBHIGF\" and \"\".join(postorder_marked(t)) == \"ACEDBHIGF\"\n    assert postorder_reversed(None) == [] and postorder_marked(None) == []\n    chain = None\n    for i in range(20_000):\n        chain = T(i, chain)\n    assert postorder_marked(chain)[:2] == [0, 1] and postorder_reversed(chain)[-1] == 19_999")

### Exercise: An in-order iterator

Write `InOrder(root)`, an iterator class producing the tree's values in in-order, one per `next()`, holding only a stack of the nodes on the current path (never a list of all the values). Then write `merge_sorted(a, b)` which, given two trees whose in-orders are sorted, returns all their values as one sorted list by walking **two** `InOrder` iterators side by side, like Week 19's merge.

In [ ]:
class TreeNode:
    def __init__(self, val, left=None, right=None):
        self.val, self.left, self.right = val, left, right

class InOrder:
    ___

def merge_sorted(a, b):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("T = TreeNode\ndef test_iter():\n    \"one at a time\"\n    t = T(4, T(2, T(1), T(3)), T(6, T(5), T(7)))\n    it = InOrder(t)\n    assert iter(it) is it and next(it) == 1 and next(it) == 2 and list(it) == [3, 4, 5, 6, 7] and list(InOrder(None)) == []\n    assert len(InOrder(t)._stack) <= 3, \"only the path\"\ndef test_merge():\n    \"merging\"\n    a = T(4, T(2, T(1), T(3)), T(6))\n    b = T(5, T(0), T(9, T(7)))\n    assert merge_sorted(a, b) == [0, 1, 2, 3, 4, 5, 6, 7, 9] and merge_sorted(None, b) == [0, 5, 7, 9] and merge_sorted(None, None) == []")

### Exercise: Rebuilding, in O(n)

Write `build(preorder, inorder)`: rebuild the binary tree (all values distinct) in **O(n)**. Use a dictionary from each value to its position in `inorder`, and a recursive helper working on index ranges instead of slices. Raise `ValueError` if the two lists don't have the same values. The checks rebuild a balanced tree of 30,000 nodes.

In [ ]:
class TreeNode:
    def __init__(self, val, left=None, right=None):
        self.val, self.left, self.right = val, left, right

def build(preorder, inorder):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("import random\nT = TreeNode\ndef pre(n):\n    return [] if n is None else [n.val] + pre(n.left) + pre(n.right)\ndef ino(n):\n    return [] if n is None else ino(n.left) + [n.val] + ino(n.right)\ndef test_build():\n    \"rebuilding\"\n    t = build(list(\"FBADCEGIH\"), list(\"ABCDEFGHI\"))\n    assert \"\".join(pre(t)) == \"FBADCEGIH\" and \"\".join(ino(t)) == \"ABCDEFGHI\" and t.right.right.left.val == \"H\"\n    assert build([], []) is None\n    try:\n        build([1, 2], [1, 3])\n        assert False\n    except ValueError:\n        pass\ndef test_big():\n    \"30,000 nodes\"\n    def balanced(lo, hi):\n        if lo > hi:\n            return None\n        mid = (lo + hi) // 2\n        return T(mid, balanced(lo, mid - 1), balanced(mid + 1, hi))\n    t = balanced(0, 29_999)\n    p, i = pre(t), ino(t)\n    again = build(p, i)\n    assert pre(again) == p and ino(again) == i")

### Exercise: Saving and loading

Write `save(root)`: a string of the tree's values in pre-order, separated by single spaces, with `#` for each missing child. And `load(text)`: rebuild the tree (values are whole numbers, possibly negative). Then `copy(root)`: an independent copy made by saving and loading. Use an explicit stack or recursion as you like (trees in the checks are at most 300 tall).

In [ ]:
class TreeNode:
    def __init__(self, val, left=None, right=None):
        self.val, self.left, self.right = val, left, right

def save(root):
    ___

def load(text):
    ___

def copy(root):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("T = TreeNode\ndef test_save():\n    \"saving\"\n    t = T(1, T(2), T(3, T(-4), T(5)))\n    assert save(t) == \"1 2 # # 3 -4 # # 5 # #\" and save(None) == \"#\"\n    again = load(save(t))\n    assert save(again) == save(t) and load(\"#\") is None\n    c = copy(t)\n    c.left.val = 99\n    assert t.left.val == 2 and c is not t\n    chain = None\n    for i in range(300):\n        chain = T(i, None, chain)\n    assert save(load(save(chain))) == save(chain)")

### Bug hunt: Right before left

This iterative pre-order pushes the **left** child first, so the right child comes off the stack first and the order is node, right, left: it prints `F G I H B D E C A`. Push the right child first, so it prints `F B A D C E G I H`.

In [ ]:
class TreeNode:
    def __init__(self, val, left=None, right=None):
        self.val, self.left, self.right = val, left, right

def preorder(root):
    out, stack = [], [root]
    while stack:
        node = stack.pop()
        out.append(node.val)
        if node.left:
            stack.append(node.left)
        if node.right:
            stack.append(node.right)
    return out

T = TreeNode
t = T("F", T("B", T("A"), T("D", T("C"), T("E"))), T("G", None, T("I", T("H"))))
print(" ".join(preorder(t)))

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_output():\n    \"left comes off first\"\n    expect_output(\"F B A D C E G I H\")")

### Bug hunt: In-order, out of order

This "in-order" visits the node before its left subtree, so it's really pre-order and the values of this search tree don't come out sorted: it prints `[4, 2, 1, 3, 6, 5, 7]`. Move the visit, so it prints `[1, 2, 3, 4, 5, 6, 7]`.

In [ ]:
class TreeNode:
    def __init__(self, val, left=None, right=None):
        self.val, self.left, self.right = val, left, right

def inorder(node, out):
    if node is None:
        return out
    out.append(node.val)
    inorder(node.left, out)
    inorder(node.right, out)
    return out

T = TreeNode
print(inorder(T(4, T(2, T(1), T(3)), T(6, T(5), T(7))), []))

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_output():\n    \"left, node, right\"\n    expect_output(\"[1, 2, 3, 4, 5, 6, 7]\")")

### Bug hunt: Split in the wrong place

When rebuilding, the left subtree has `k` values, where `k` is the root's position in the **in-order**. This version takes the left part of the pre-order as `preorder[1:k]`, one too few, so the tree comes out wrong and this prints `False`. Fix both slices so the rebuilt tree's in-order and pre-order match the originals, and it prints `True`.

In [ ]:
class TreeNode:
    def __init__(self, val, left=None, right=None):
        self.val, self.left, self.right = val, left, right

def build(preorder, inorder):
    if not preorder:
        return None
    k = inorder.index(preorder[0])
    return TreeNode(preorder[0], build(preorder[1:k], inorder[:k]), build(preorder[k:], inorder[k + 1:]))

def pre(n):
    return [] if n is None else [n.val] + pre(n.left) + pre(n.right)

def ino(n):
    return [] if n is None else ino(n.left) + [n.val] + ino(n.right)

t = build(list("FBADCEGIH"), list("ABCDEFGHI"))
print("".join(pre(t)) == "FBADCEGIH" and "".join(ino(t)) == "ABCDEFGHI")

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_output():\n    \"split by size\"\n    expect_output(\"True\")")

### Puzzle: In-order with a stack

Arrange the lines into an iterative in-order traversal. One line is a trap.

(Put these lines in order on the website.)

### Exercise: Predict them



In [ ]:
# For the tree         1
#                     / \
#                    2   3
#                   / \   \
#                  4   5   6

p = ___   # the pre-order, as a list
q = ___   # the in-order
r = ___   # the post-order
s = ___   # save() of this tree with "#" markers, as in today's practice
t = ___   # the most nodes on the explicit stack at once during the iterative in-order

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_values():\n    \"each\"\n    assert p == [1, 2, 4, 5, 3, 6] and q == [4, 2, 5, 1, 3, 6] and r == [4, 5, 2, 6, 3, 1], \"p, q and r\"\n    assert s == \"1 2 4 # # 5 # # 3 # 6 # #\" and t == 3, \"s and t\"")

### Quiz

Take this quiz on the website: it is marked for you and feeds your daily review.

### Exercise: Challenge: flattening and common ancestors

Two classic problems:

1. `flatten(root)`: rearrange the tree **in place** into a "linked list" that follows pre-order, using the `right` pointers, with every `left` set to `None`. Do it with an explicit stack (the checks use a tree 5,000 tall). Return the root.
2. `common_ancestor(root, a, b)`: the lowest node that has both values `a` and `b` in its subtree (a node counts as in its own subtree), in a tree **without** parent links. One post-order pass: a node is the answer if `a` and `b` are found in different subtrees, or it is one of them and the other is below it. Return the node's value, or `None` if either value is missing. Values are distinct.

In [ ]:
class TreeNode:
    def __init__(self, val, left=None, right=None):
        self.val, self.left, self.right = val, left, right

def flatten(root):
    ___

def common_ancestor(root, a, b):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("T = TreeNode\ndef pre(n):\n    out, stack = [], [n] if n else []\n    while stack:\n        x = stack.pop()\n        out.append(x.val)\n        if x.right:\n            stack.append(x.right)\n        if x.left:\n            stack.append(x.left)\n    return out\ndef test_flatten():\n    \"flattening\"\n    t = T(1, T(2, T(3), T(4)), T(5, None, T(6)))\n    order = pre(t)\n    f = flatten(t)\n    vals, node = [], f\n    while node is not None:\n        assert node.left is None\n        vals.append(node.val)\n        node = node.right\n    assert vals == order == [1, 2, 3, 4, 5, 6] and flatten(None) is None\n    chain = None\n    for i in range(5000):\n        chain = T(i, chain, None)\n    f = flatten(chain)\n    count, node = 0, f\n    while node is not None:\n        count += 1\n        node = node.right\n    assert count == 5000\ndef test_lca():\n    \"common ancestors\"\n    t = T(3, T(5, T(6), T(2, T(7), T(4))), T(1, T(0), T(8)))\n    assert common_ancestor(t, 5, 1) == 3 and common_ancestor(t, 5, 4) == 5 and common_ancestor(t, 7, 4) == 2 and common_ancestor(t, 6, 8) == 3\n    assert common_ancestor(t, 6, 99) is None and common_ancestor(t, 0, 0) == 0")

## Warm-up

The same walk, with the "visit" in three different places, prints the expression `(2 + 3) * 4` in prefix (Polish), infix (without its brackets) and postfix (reverse Polish, Week 43) notation. Where you put the visit is the only difference between the three depth-first orders.

## Three orders

```text
            F
          /   \
         B     G            pre-order:   F B A D C E G I H     node, then left, then right
        / \     \           in-order:    A B C D E F G H I     left, then node, then right
       A   D     I          post-order:  A C E D B H I G F     left, then right, then node
          / \   /
         C   E H
```

| Order | Visit | Typical jobs |
|---|---|---|
| **pre-order** | node, left, right | copying a tree; saving it (the root comes first, so it can be rebuilt as it's read); printing an outline; prefix expressions |
| **in-order** | left, node, right | in a binary **search** tree (next week), visits values in sorted order; infix expressions |
| **post-order** | left, right, node | anything that needs the children finished first: sizes, heights, deleting a tree, evaluating an expression, postfix |

Yesterday's `size` and `height` were post-order (both children's answers first, then the node's); `mirror` could be either pre- or post-order. Each traversal visits every node once: O(n) time, and O(h) extra space for the recursion, where h is the height.

## With an explicit stack

Recursion uses Python's call stack, which is limited to about 1,000 frames; a tall tree (a long chain) breaks it. An explicit stack (Week 43) has no such limit. Each order needs a slightly different loop.

**Pre-order** is the easiest: pop a node, visit it, push its right child and then its left (so the left comes off first).

**In-order**: go as far left as possible, pushing each node; when you can't go further, pop a node, visit it, and move to its right child, which starts the left-going again.

**Post-order**: a node must wait until both children are done. A neat trick: a pre-order walk that pushes **left before right** visits node, right, left, which is exactly post-order **reversed**.

## One at a time

Sometimes you want a tree's values **one at a time**, without building the whole list: to stop early, or to walk two trees side by side. Week 36's iterator classes do exactly that. The in-order loop above splits neatly into "push the left spine" and "pop one, then push the left spine of its right child":

Each `next` does O(1) work on average (every node is pushed and popped once in all), and the iterator only ever holds O(h) nodes: the path down to where it is. (Phase 5's generators make iterators like this much shorter to write.)

## Rebuilding a tree

A single traversal doesn't pin down a tree's shape: many trees have the same in-order list. But **pre-order plus in-order** does, as long as the values are all different. The first value of the pre-order is the root; finding it in the in-order splits that list into the left subtree (before it) and the right (after it), and the sizes tell you how to split the pre-order too:

This is O(n²) in the worst case, because of `index` and the slicing; a dictionary from value to in-order position, and passing index ranges instead of slices, makes it O(n) (you'll do that in the practice).

## Saving a tree

To write a tree to a file and read it back exactly, walk it in **pre-order** and write a marker (say `#`) for every missing child. Reading it back is the same walk: take the next token; `#` means "no node here", anything else makes a node, whose left and then right subtrees come next.

`position` is a one-item list so that every call moves the **same** reading position forward (Phase 5's `nonlocal` is the tidier way). With the markers, the pre-order alone is enough to rebuild the shape.

## Practice

## Quiz

## Daily challenge

## Before you go

Take the tree from "Three orders" and, without looking, write its pre-, in- and post-order. Then check them with the code. Tomorrow: breadth-first traversal, level by level, with a queue.